# Entrainement de la Detection

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from biomae.paths import get_project_root
PROJECT_ROOT = get_project_root()

from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os
import numpy as np # Nécessaire pour manipuler les chiffres

path = "PROJECT_ROOT/"

def main():
    # 1. Charger le modèle
    model = YOLO("yolo11n.pt") 

    # 2. Lancer l'entraînement
    results = model.train(
        # Chemin absolu vers le data.yaml (Parfait)
        data = f"{path}data/final_split/data.yaml",
        
        # --- Paramètres de performance ---
        epochs=100,
        imgsz=1024,
        # Batch=2 est très sécuritaire (low VRAM). Si tu as une bonne carte (ex: 3060/4060), 
        # tu pourras probablement tenter batch=4 ou 8 au prochain run.
        batch=2,
        
        # --- Paramètres Pragmatiques ---
        single_cls=True,
        
        # --- Augmentation ---
        mosaic=1.0,
        mixup=0.1,
        degrees=15.0,
        
        # --- Hardware ---
        device=0,
        workers=8,
        
        # --- CORRECTION ICI ---
        # 'project' : Le dossier racine où tout sera stocké (Chemin absolu OK)
        project=f"{path}outputs/yolov11n_i1024b2_e100",
        
        # 'name' : Juste le nom du sous-dossier final (PAS de chemin absolu)
        name="det", 
        
        exist_ok=True
    )

    # 3. Validation
    metrics = model.val()
    print(f"mAP50: {metrics.box.map50}")

if __name__ == '__main__':
    main()

In [ ]:
%matplotlib inline
path_to_model = f"{path}outputs/yolov11n_i1024b2_e100/det/weights/best.pt"
# Vérification simple pour ne pas perdre de temps
if not os.path.exists(path_to_model):
    print(f"⚠️ ATTENTION : Modèle introuvable ici : {path_to_model}")
    print("Utilise 'yolo11n.pt' pour tester si ton entraînement n'est pas fini.")
    model = YOLO("yolo11n.pt") # Fallback sur le modèle de base
else:
    model = YOLO(path_to_model)

# 2. Choisir une image de test
image_path = f"{path}data/final_split/val/images/img_A_00012.png"

# 3. Lancer l'inférence
# conf=0.25 : Seuil de confiance (ajuste si tu ne détectes rien)
# save=True : Sauvegarde aussi l'image sur le disque (dans runs/detect/predict)
# 3. Lancer l'inférence avec réglage NMS
results = model.predict(
    source=image_path, 
    conf=0.25,      # Seuil de confiance (garde ce qui est sûr)
    iou=0.6,        # <--- C'EST ICI. Par défaut c'est 0.7, mais tu peux le monter.
    save=False
)
# 2. Lance la validation
metrics = model.val()

# 3. Affiche les specs techniques
print(f"--- SPECS DU MODÈLE ---")
print(f"mAP50 (Précision globale) : {metrics.box.map50:.3f}")
print(f"mAP50-95 (Précision fine) : {metrics.box.map:.3f}")
print(f"Précision (Precision)     : {metrics.box.mp:.3f}")
print(f"Rappel (Recall)           : {metrics.box.mr:.3f}")

# 4. Afficher le résultat
for result in results:
    # result.plot() génère une matrice numpy (image) avec les boîtes dessinées
    # Par défaut c'est du BGR (format OpenCV), il faut le passer en RGB pour Matplotlib
    res_plotted = result.plot()
    
    # Conversion BGR -> RGB
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
    
    # Affichage avec Matplotlib
    plt.figure(figsize=(10, 10)) # Taille de l'affichage
    plt.imshow(res_rgb)
    plt.axis('off') # Enlève les axes moches
    plt.show()